# Lab 5 - IT443  
## Virat Shrimali - 202303061

# Bootstrap Confidence Intervals and Hypothesis Testing

## Problem Statement

In this lab, we study **resampling techniques** to estimate uncertainty and perform hypothesis testing.

### Objectives:
- Construct different types of confidence intervals:
  - Normal CI
  - Bootstrap-t CI
  - Percentile CI
  - BCa CI
- Perform hypothesis testing using:
  - Bootstrap test
  - Permutation test
  - Studentized bootstrap test

We compare methods based on:
- Accuracy
- Robustness
- Handling of skewness and bias

## Methodology

### 1. Bootstrap Principle

We approximate the unknown distribution using the empirical distribution:

$$
F_n(x) = \frac{1}{n} \sum_{i=1}^n 1\{X_i \le x\}
$$

Bootstrap samples are generated as:

$$
X_1^*, ..., X_n^* \sim F_n
$$

---

### 2. Confidence Intervals

- **Normal CI**
$$
\hat{\theta} \pm z_{\alpha/2} \cdot \hat{SE}
$$

- **Percentile CI**
Uses quantiles of bootstrap distribution

- **Bootstrap-t CI**
$$
\frac{\hat{\theta}^* - \hat{\theta}}{\hat{SE}^*}
$$

- **BCa CI**
Adjusts for:
  - Bias
  - Skewness

---

### 3. Hypothesis Testing

We test:

$$
H_0: \mu_1 = \mu_2
$$

Methods:
- Bootstrap (resampling from pooled data)
- Permutation (shuffle labels)
- Studentized bootstrap (variance normalized)

## Imports

In [54]:

import numpy as np
from sklearn.datasets import load_wine
from scipy.stats import norm


## Load Dataset

In [55]:
data = load_wine()
X = data.data[:, 0]
Y = data.data[:, 1]

n = len(X)
theta_hat = np.mean(X)

B = 2000

## Bootstrap Sampling

In [56]:
boot_means = []
boot_se = []

for _ in range(B):
    sample = np.random.choice(X, size=n, replace=True)
    boot_means.append(np.mean(sample))
    boot_se.append(np.std(sample, ddof=1)/np.sqrt(n))

boot_means = np.array(boot_means)
boot_se = np.array(boot_se)

## Confidence Intervals

In [57]:
# Normal CI
se_hat = np.std(X, ddof=1)/np.sqrt(n)
z = norm.ppf(0.975)

normal_ci = (theta_hat - z*se_hat, theta_hat + z*se_hat)
normal_ci

(np.float64(12.881356184655965), np.float64(13.119879770400216))

In [58]:
# Percentile CI
percentile_ci = np.percentile(boot_means, [2.5, 97.5])
percentile_ci


array([12.88688062, 13.12152247])

In [59]:
# Bootstrap-t CI
t_stats = (boot_means - theta_hat)/boot_se
t_low, t_high = np.percentile(t_stats, [97.5, 2.5])

boot_t_ci = (theta_hat - t_low*se_hat, theta_hat - t_high*se_hat)
boot_t_ci


(np.float64(12.876365503150327), np.float64(13.117990393689915))

In [60]:
# BCa CI
z0 = norm.ppf(np.mean(boot_means < theta_hat))
alpha1 = norm.cdf(2*z0 + norm.ppf(0.025))
alpha2 = norm.cdf(2*z0 + norm.ppf(0.975))

bca_ci = np.percentile(boot_means, [alpha1*100, alpha2*100])
bca_ci


array([12.88275194, 13.118917  ])

## Interpretation (Confidence Intervals)

- Normal CI assumes symmetry → may be inaccurate
- Percentile CI adapts to bootstrap distribution
- Bootstrap-t accounts for variability → more stable
- BCa corrects both bias and skewness → most reliable

### Key Insight:
BCa and Bootstrap-t provide better estimates for non-normal data.

## Table of Confidence Intervals

In [61]:
ci_table = [
    ["Normal CI", normal_ci[0], normal_ci[1], normal_ci[1]-normal_ci[0]],
    ["Percentile CI", percentile_ci[0], percentile_ci[1], percentile_ci[1]-percentile_ci[0]],
    ["Bootstrap-t CI", boot_t_ci[0], boot_t_ci[1], boot_t_ci[1]-boot_t_ci[0]],
    ["BCa CI", bca_ci[0], bca_ci[1], bca_ci[1]-bca_ci[0]],
]

import pandas as pd
df_ci = pd.DataFrame(ci_table, columns=["Method", "Lower", "Upper", "Length"])
df_ci

,Method,Lower,Upper,Length
0,Normal CI,12.881356,13.119880,0.238524
1,Percentile CI,12.886881,13.121522,0.234642
2,Bootstrap-t CI,12.876366,13.117990,0.241625
3,BCa CI,12.882752,13.118917,0.236165


## Comparison of CI Lengths

- The length of a confidence interval is:
$$
\text{Length} = \text{Upper} - \text{Lower}
$$

### Observations:

- Normal CI:
  - Symmetric
  - May be inaccurate if distribution is skewed

- Percentile CI:
  - Adapts to empirical distribution
  - Captures asymmetry

- Bootstrap-t CI:
  - Adjusts for variability using standard error
  - Typically more stable than percentile

- BCa CI:
  - Corrects both bias and skewness
  - Often provides the most accurate interval

### Key Insight:
Shorter interval ≠ better  
→ Correct coverage is more important than length

---
## Skewness and Bias Correction

- If the bootstrap distribution is symmetric:
  - All methods give similar results

- If the distribution is skewed:
  - Normal CI becomes inaccurate (assumes symmetry)
  - Percentile CI shifts according to data
  - BCa corrects both:
    - Bias (location shift)
    - Skewness (asymmetric adjustment)

### Final Comment:

- BCa is the most reliable method when:
  - Data is skewed
  - Sample size is moderate

- Bootstrap-t also performs well due to studentization


# Problem 10: Hypothesis Testing

Test:
$H_{0}$: μ1 = μ2

Methods:
- Bootstrap test
- Permutation test
- Studentized bootstrap


## Methodology: Hypothesis Testing

We test:

$$
H_0: \mu_1 = \mu_2
\quad \text{vs} \quad
H_1: \mu_1 \neq \mu_2
$$

Let:
- $X_1, ..., X_m \sim F$
- $Y_1, ..., Y_n \sim G$

Define the observed statistic:

$$
T = \bar{X} - \bar{Y}
$$

---

### 1. Bootstrap Test

#### Idea:
Simulate the null distribution by resampling from pooled data.

#### Steps:

1. Pool the data:
$$
Z = \{X_1, ..., X_m, Y_1, ..., Y_n\}
$$

2. Generate bootstrap samples:
$$
Z^* \sim \hat{F}_n
$$

3. Split into two groups:
- First $m$ → $X^*$
- Remaining $n$ → $Y^*$

4. Compute:
$$
T^* = \bar{X}^* - \bar{Y}^*
$$

5. Repeat B times to get distribution of $T^*$

6. Compute p-value (Achieved Significance Level):

$$
\text{ASL} = P(|T^*| \ge |T|)
$$

---

### 2. Permutation Test

#### Idea:
Under $H_0$, labels are exchangeable.

#### Steps:

1. Combine samples:
$$
Z = \{X, Y\}
$$

2. Randomly permute labels

3. Split into two groups:
- First $m$ → $X^*$
- Remaining $n$ → $Y^*$

4. Compute:
$$
T^* = \bar{X}^* - \bar{Y}^*
$$

5. Repeat B times

6. Compute p-value:
$$
p = P(|T^*| \ge |T|)
$$

#### Key Property:
Permutation test is **exact** under exchangeability.

---

### 3. Studentized Bootstrap Test

#### Idea:
Normalize statistic to account for variability.

#### Statistic:

$$
T = \frac{\bar{X} - \bar{Y}}{\sqrt{\frac{\hat{\sigma}_1^2}{m} + \frac{\hat{\sigma}_2^2}{n}}}
$$

#### Steps:

1. Compute observed studentized statistic

2. Bootstrap under $H_0$

3. For each sample compute:

$$
T^* = \frac{\bar{X}^* - \bar{Y}^*}{\sqrt{\frac{\hat{\sigma}_1^{*2}}{m} + \frac{\hat{\sigma}_2^{*2}}{n}}}
$$

4. Compute p-value:
$$
p = P(|T^*| \ge |T|)
$$

#### Advantage:
- Accounts for unequal variances
- More stable than raw bootstrap

## Observed Statistic

- We construct two independent samples using the same feature across different classes to ensure a valid two-sample comparison.

In [62]:
from sklearn.datasets import load_wine

data = load_wine()

X_full = data.data[:, 0]   # choose ONE feature (e.g., alcohol)
labels = data.target

# Two independent groups
X = X_full[labels == 0]
Y = X_full[labels == 1]

n1, n2 = len(X), len(Y)

## Bootstrap test

In [63]:
pooled = np.concatenate([X, Y])

boot_stats = []
for _ in range(B):
    sample = np.random.choice(pooled, size=n1+n2, replace=True)
    s1, s2 = sample[:n1], sample[n1:]
    boot_stats.append(np.mean(s1) - np.mean(s2))

boot_stats = np.array(boot_stats)
p_boot = (np.sum(np.abs(boot_stats) >= abs(obs_stat)) + 1) / (B + 1)
p_boot

np.float64(0.0004997501249375312)

## Permutation test

In [64]:
perm_stats = []
for _ in range(B):
    perm = np.random.permutation(pooled)
    s1, s2 = perm[:n1], perm[n1:]
    perm_stats.append(np.mean(s1) - np.mean(s2))

perm_stats = np.array(perm_stats)
p_perm = (np.sum(np.abs(perm_stats) >= abs(obs_stat)) + 1) / (B + 1)
p_perm

np.float64(0.0004997501249375312)

## Studentized Bootstrap

In [65]:
stud_stats = []

for _ in range(B):
    sample = np.random.choice(pooled, size=n1+n2, replace=True)
    s1, s2 = sample[:n1], sample[n1:]

    num = np.mean(s1) - np.mean(s2)
    den = np.sqrt(np.var(s1)/n1 + np.var(s2)/n2)

    stud_stats.append(num/den)

stud_stats = np.array(stud_stats)

obs_stud = obs_stat / np.sqrt(np.var(X)/n1 + np.var(Y)/n2)

p_stud = (np.sum(np.abs(stud_stats) >= abs(obs_stud)) + 1) / (B + 1)
p_stud

np.float64(0.0004997501249375312)

## Table of p-values

In [66]:
pval_table = [
    ["Bootstrap Test", p_boot],
    ["Permutation Test", p_perm],
    ["Studentized Bootstrap", p_stud],
]

df_pval = pd.DataFrame(pval_table, columns=["Method", "p-value"])
df_pval

,Method,p-value
0,Bootstrap Test,0.0005
1,Permutation Test,0.0005
2,Studentized Bootstrap,0.0005


## Interpretation of Differences

- Bootstrap Test:
  - Uses resampling from pooled data
  - Provides approximate p-values
  - May be sensitive to sample variability

- Permutation Test:
  - Based on label shuffling
  - Exact under $H_0$
  - Does not rely on distributional assumptions

- Studentized Bootstrap:
  - Normalizes statistic using variance
  - More stable and robust
  - Handles unequal variances better

### Key Insight:

- Permutation test is most reliable when exchangeability holds
- Studentized bootstrap improves over naive bootstrap

## Robustness and Assumptions

### Bootstrap Test
- Assumes sample represents population
- Works without strict distribution assumptions
- May be biased for small samples

---

### Permutation Test
- Assumes exchangeability under $H_0$
- No parametric assumptions
- Provides exact inference

---

### Studentized Bootstrap
<!-- - Accounts for heteroskedasticity -->
- More robust to variance differences
- Preferred in real-world data scenarios

---

### Final Conclusion

- Permutation → best when valid (exact)
- Studentized bootstrap → best practical alternative
- Plain bootstrap → flexible but less reliable